In [4]:
import sys
sys.path.append("../src")
import pandas as pd
import numpy as np
from evaluate import ablation
from models import get_feature_cols, train_cv

X = pd.read_parquet("../data/processed/features.parquet")
print("Loaded:", X.shape)

rotation_feats = ["yaw_gcd", "yaw_std", "yaw_kurtosis", "yaw_skew", "yaw_max",
                   "yaw_p99", "yaw_pitch_ratio", "pitch_zero_frac",
                   "snap_count", "reversal_rate"]
timing_feats = ["cps_mean", "iv_std", "iv_cv", "iv_kurtosis", "iv_skew",
                 "iv_min", "iv_unique_frac", "iv_p95_p50", "iv_mean_median"]
reach_feats = ["reach_mean", "reach_std", "reach_p95", "reach_max",
               "reach_over_30", "reach_over_29", "hit_rate", "ping_mean"]
movement_feats = ["speed_resid_mean", "speed_resid_max", "speed_resid_pos",
                   "speed_max", "airborne_frac", "fall_resid_std",
                   "fall_resid_max", "air_const_y_frac"]

all_features = get_feature_cols(X)

drop_groups = {
    "rotation": rotation_feats,
    "timing": timing_feats,
    "reach": reach_feats,
    "movement": movement_feats,
}

ablation_table = ablation(X, all_features, train_cv, drop_groups)
print(ablation_table)

# Now train the full model once more and save its oof predictions for the dashboard
oof, models = train_cv(X)
np.save("../data/processed/oof_scores.npy", oof)
print("Saved oof_scores.npy, shape:", oof.shape)

Loaded: (369, 40)
[LightGBM] [Info] Number of positive: 120, number of negative: 175
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001660 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2139
[LightGBM] [Info] Number of data points in the train set: 295, number of used features: 35
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.406780 -> initscore=-0.377294
[LightGBM] [Info] Start training from score -0.377294
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with 

In [5]:
import sys
sys.path.append("../src")
import pandas as pd
from evaluate import evaluate

X = pd.read_parquet("../data/processed/features.parquet")

ap, threshold_table = evaluate(X.is_cheat.values, oof)
print("AP:", ap)
print(threshold_table)

AP: 0.7775655884009292
   threshold  precision    recall       fpr  flagged
0       0.50   0.620690  0.755245  0.292035      174
1       0.70   0.649682  0.713287  0.243363      157
2       0.80   0.655844  0.706294  0.234513      154
3       0.90   0.666667  0.657343  0.207965      141
4       0.95   0.682171  0.615385  0.181416      129
5       0.99   0.784314  0.559441  0.097345      102


In [6]:
from evaluate import recall_by_type
recall_table = recall_by_type(X, oof, threshold=0.90)
print(recall_table)

         cheat   n    recall
0  AUTOCLICKER  41  0.878049
1     KILLAURA  44  1.000000
2        REACH  58  0.241379


In [7]:
sessions = pd.read_csv("../server/plugins/Collector/sessions.csv")
sessions = sessions[sessions.label != "UNLABELED"]
print(sessions.groupby("label").size())
print(sessions.player.nunique(), "players")
print((sessions.ended_ts - sessions.started_ts).sum() / 1000 / 3600, "total hours")

label
AUTOCLICKER     6
CLICKTEST       1
KILLAURA        2
LEGIT           8
REACH           6
ROTATIONTEST    1
TEST            1
TEST1           1
dtype: int64
1 players
2.3543105555555557 total hours


In [8]:
ablation_table = ablation(X, all_features, train_cv, drop_groups)
print(ablation_table)

[LightGBM] [Info] Number of positive: 120, number of negative: 175
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000267 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2139
[LightGBM] [Info] Number of data points in the train set: 295, number of used features: 35
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.406780 -> initscore=-0.377294
[LightGBM] [Info] Start training from score -0.377294
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, bes

In [9]:
print(ablation_table.to_string())

       removed        ap
0         full  0.777566
1  no_rotation  0.685014
2    no_timing  0.627156
3     no_reach  0.818312
4  no_movement  0.735543
